In [1]:
"""
Second-dataset generalization check: does the pseudo-replication effect
found in the Kapranova/Chen exchange-bias data also appear in an
independent literature-compiled dataset (Kim et al., Small 2023)?

Files needed in the same folder:
    r1.csv, r2.csv, SAR.csv

Requirements: pip install pandas numpy scikit-learn
"""
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import KBinsDiscretizer

MODELS = {
    "RandomForest":         RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42, max_iter=150),
}

DATASETS = {
    "r1 relaxivity (MRI T1)": {"file": "r1.csv",  "target": "r1 relaxivity", "group_col": "Core composition"},
    "r2 relaxivity (MRI T2)": {"file": "r2.csv",  "target": "r2 relaxivity", "group_col": "Core composition"},
    "SAR (hyperthermia)":     {"file": "SAR.csv", "target": "SAR",           "group_col": "Core composition"},
}


def describe_dataset(df, group_col, name):
    """Basic pseudo-replication diagnostics, mirroring what was done for the
    exchange-bias dataset."""
    n_rows = len(df)
    n_unique = df[group_col].nunique()
    top = df[group_col].value_counts()
    dominant_frac = 100 * top.iloc[0] / n_rows

    print(f"\n--- {name} ---")
    print(f"  Rows: {n_rows}")
    print(f"  Unique compositions ({group_col}): {n_unique}")
    print(f"  Most repeated composition: '{top.index[0]}' -- {top.iloc[0]} rows "
          f"({dominant_frac:.1f}% of the whole dataset)")
    print(f"  Top 5 compositions by row count:")
    print(top.head(5).to_string())
    return n_rows, n_unique, dominant_frac


def empirical_leakage_check(df, group_col, n_folds=5):
    """Same check used on the exchange-bias data: what fraction of test rows
    share their exact composition with the training set under standard
    shuffled K-Fold."""
    groups = df[group_col].values
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    total_test, total_leaked = 0, 0
    for train_idx, test_idx in kf.split(np.arange(len(df))):
        train_groups = set(groups[train_idx])
        leaked = np.isin(groups[test_idx], list(train_groups))
        total_test += len(test_idx)
        total_leaked += leaked.sum()
    pct = 100 * total_leaked / total_test
    print(f"  Empirical leakage: {total_leaked}/{total_test} test rows ({pct:.1f}%) "
          f"share their exact composition with training under shuffled 5-fold CV.")
    return pct


def evaluate_leaky_vs_honest(df, target_col, group_col, n_folds=5):
    """Identical protocol to the main exchange-bias benchmark: compare
    shuffled KFold ('leaky') to GroupKFold by composition ('honest')."""
    d = df.dropna(subset=[target_col]).copy()
    feature_cols = [c for c in d.columns if c not in
                     [target_col, "Core composition", "Shell composition", group_col]
                     and d[c].dtype != object]
    X = d[feature_cols].values
    y = d[target_col].values
    y = np.log10(np.clip(y, a_min=1e-6, a_max=None))  # log-transform, matches the user's own r1/r2/SAR scripts

    Xs = MinMaxScaler().fit_transform(X)
    groups = d[group_col].values

    splitters = {
        "shuffled_kfold (leaky)":       (KFold(n_splits=n_folds, shuffle=True, random_state=42), {}),
        "groupkfold_by_composition (honest)": (GroupKFold(n_splits=n_folds), dict(groups=groups)),
    }

    rows = []
    for split_name, (splitter, kwargs) in splitters.items():
        for model_name, model in MODELS.items():
            cv = cross_validate(model, Xs, y, cv=splitter, scoring=["r2"], n_jobs=-1, **kwargs)
            rows.append({"split": split_name, "model": model_name,
                        "r2_mean": round(cv["test_r2"].mean(), 3),
                        "r2_std": round(cv["test_r2"].std(), 3)})
    return pd.DataFrame(rows), len(feature_cols)


if __name__ == "__main__":
    print("=" * 78)
    print("SECOND-DATASET PSEUDO-REPLICATION CHECK (Kim et al., Small 2023)")
    print("=" * 78)

    all_results = []
    for name, cfg in DATASETS.items():
        df = pd.read_csv(cfg["file"])
        describe_dataset(df, cfg["group_col"], name)
        empirical_leakage_check(df, cfg["group_col"])

        print(f"  Running leaky-vs-honest comparison for target '{cfg['target']}' ...")
        res, n_feat = evaluate_leaky_vs_honest(df, cfg["target"], cfg["group_col"])
        res["dataset"] = name
        print(f"  (using {n_feat} numeric feature columns)")
        print(res.to_string(index=False))
        all_results.append(res)

    full = pd.concat(all_results, ignore_index=True)
    full = full[["dataset", "split", "model", "r2_mean", "r2_std"]]
    full.to_csv("results_second_dataset_check.csv", index=False)

    print("\n" + "=" * 78)
    print("SUMMARY -- compare these gaps to the exchange-bias dataset's 0.24-0.30 R2 drop")
    print("=" * 78)
    print(full.to_string(index=False))
    print("\nSaved to results_second_dataset_check.csv")

SECOND-DATASET PSEUDO-REPLICATION CHECK (Kim et al., Small 2023)

--- r1 relaxivity (MRI T1) ---
  Rows: 355
  Unique compositions (Core composition): 67
  Most repeated composition: 'Fe1Fe2O4' -- 122 rows (34.4% of the whole dataset)
  Top 5 compositions by row count:
Core composition
Fe1Fe2O4             122
Mn1Fe2O4              27
Na1Ho1F4              17
Mn0.43Fe0.57Fe2O4     16
Fe2O3                 16
  Empirical leakage: 318/355 test rows (89.6%) share their exact composition with training under shuffled 5-fold CV.
  Running leaky-vs-honest comparison for target 'r1 relaxivity' ...
  (using 8 numeric feature columns)
                             split                model  r2_mean  r2_std                dataset
            shuffled_kfold (leaky)         RandomForest    0.732   0.036 r1 relaxivity (MRI T1)
            shuffled_kfold (leaky) HistGradientBoosting    0.715   0.039 r1 relaxivity (MRI T1)
groupkfold_by_composition (honest)         RandomForest    0.272   0.279 r1 rel